In [ ]:
import os

WORKSHOP_RESOURCE_GROUP = os.getenv("WORKSHOP_RESOURCE_GROUP", "YOUR_RESOURCE_GROUP").strip()
WORKSHOP_AUTH_MODE = os.getenv("WORKSHOP_AUTH_MODE", "managed-identity").strip()
os.environ["WORKSHOP_RESOURCE_GROUP"] = WORKSHOP_RESOURCE_GROUP
os.environ["WORKSHOP_AUTH_MODE"] = WORKSHOP_AUTH_MODE
os.environ["CHAT_DEPLOYMENT_NAME"] = ""
os.environ["CHAT_MODEL_NAME"] = ""
os.environ["EMBEDDING_DEPLOYMENT_NAME"] = ""
os.environ["EMBEDDING_MODEL_NAME"] = ""

print(f"Notebook config: WORKSHOP_RESOURCE_GROUP={WORKSHOP_RESOURCE_GROUP}, WORKSHOP_AUTH_MODE={WORKSHOP_AUTH_MODE}")
print("Set CHAT_DEPLOYMENT_NAME, CHAT_MODEL_NAME, EMBEDDING_DEPLOYMENT_NAME, EMBEDDING_MODEL_NAME in this kernel.")

In [ ]:
import os
import shlex
import subprocess

cmd = [
    "bash",
    "../../scripts/assign-workshop-env.sh",
    "--resource-group",
    WORKSHOP_RESOURCE_GROUP,
    "--auth-mode",
    WORKSHOP_AUTH_MODE,
]

result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stderr.strip():
    print(result.stderr.strip())

for line in result.stdout.splitlines():
    line = line.strip()
    if not line.startswith("export "):
        continue
    key, raw_value = line[len("export "):].split("=", 1)
    parsed = shlex.split(raw_value)
    os.environ[key] = parsed[0] if parsed else ""

print("Workshop environment variables loaded into notebook kernel. Azure token auth uses AzureCliCredential via az login.")

In [ ]:
import importlib
import sys
from pathlib import Path

NOTEBOOK_PATH_CANDIDATES = [Path.cwd(), Path.cwd() / "AgentWorkshop" / "Notebook"]
for candidate in NOTEBOOK_PATH_CANDIDATES:
    if (candidate / "workshop_bootstrap.py").exists():
        resolved_candidate = str(candidate.resolve())
        if resolved_candidate not in sys.path:
            sys.path.insert(0, resolved_candidate)

import workshop_bootstrap
importlib.reload(workshop_bootstrap)
build_workshop_config = workshop_bootstrap.build_workshop_config

CONFIG_OVERRIDES = {
    "resource_group_name": "",
    "location": "",
    "subscription_id": "",
    "foundry_account_name": "",
    "foundry_project_name": "",
    "foundry_project_endpoint": "",
    "foundry_project_api_key": "",
    "search_service_name": "",
    "search_api_key": "",
    "storage_account_name": "",
    "application_insights_name": "",
    "model_zone": "",
}

config = build_workshop_config(CONFIG_OVERRIDES)
config.show()

# Workshop 2: Foundry IQ Knowledge Sources And Knowledge Base

This notebook mirrors docs/knowledge_base.md and creates Foundry IQ objects with code.

Flow:
1. Parse workshop files from data/Coffee
2. Create Search indexes
3. Create Foundry IQ knowledge sources over those indexes
4. Create a knowledge base that binds the knowledge sources and chat model
5. Output MCP endpoint for downstream agents

In [ ]:
# Uncomment this cell in a clean kernel.
# %pip install --quiet requests azure-identity pypdf

In [ ]:
import csv
import os
from pathlib import Path

from pypdf import PdfReader

from workshop_bootstrap import (
    WorkshopConstants,
    build_search_client,
    create_knowledge_base,
    create_knowledge_source,
    create_search_index,
    sanitize_name,
    upload_documents,
)

DATA_ROOT = Path("../../data/Coffee").resolve()
MAX_PDF_CHARACTERS = 12000
MAX_CSV_ROWS = 500
MAX_CSV_FIELDS = 20
KNOWLEDGE_BASE_NAME = "health-effects-kb"

CHAT_DEPLOYMENT_NAME = os.getenv("CHAT_DEPLOYMENT_NAME", "").strip()
CHAT_MODEL_NAME = os.getenv("CHAT_MODEL_NAME", CHAT_DEPLOYMENT_NAME).strip()
AZURE_OPENAI_ENDPOINT = (os.getenv("AZURE_OPENAI_ENDPOINT", "").strip() or config.openai_resource_endpoint).rstrip("/")

if not CHAT_DEPLOYMENT_NAME:
    raise ValueError("Set CHAT_DEPLOYMENT_NAME in your environment before running this cell.")
if not CHAT_MODEL_NAME:
    raise ValueError("Set CHAT_MODEL_NAME in your environment before running this cell.")
if not AZURE_OPENAI_ENDPOINT:
    raise ValueError("AZURE_OPENAI_ENDPOINT or AZURE_AI_PROJECT_ENDPOINT must be set.")

SOURCE_SPECS = [
    {"knowledge_source_name": "health-effects-ks", "folder": DATA_ROOT / "HealthEffects", "source_type": "health-effects"},
    {"knowledge_source_name": "coffee-recipes-ks", "folder": DATA_ROOT / "CoffeeRecipes", "source_type": "coffee-recipes"},
    {"knowledge_source_name": "coffee-csv-generalhealth-ks", "folder": DATA_ROOT / "CoffeeCSV" / "GeneralHealth", "source_type": "coffee-csv-generalhealth"},
    {"knowledge_source_name": "coffee-csv-mentalhealth-ks", "folder": DATA_ROOT / "CoffeeCSV" / "mentalHealth", "source_type": "coffee-csv-mentalhealth"},
]

KB_RETRIEVAL_INSTRUCTIONS = """
You are a scientific retrieval assistant for a coffee-and-health knowledge base.

Goal:
Return evidence-grounded, citation-rich answers suitable for academic and research workflows. Prefer high-quality sources, explicitly state uncertainty, and avoid unsupported claims.

Knowledge source routing:
1. HealthEffects (PDF studies/reviews):
- Primary source for health outcomes, mechanisms, risks, and benefits.
- Use first for any clinical, epidemiologic, or biomedical question.

2. CoffeeCSV/GeneralHealth (synthetic CSV):
- Use for exploratory patterns in lifestyle, mood, stress, and coffee intake.
- Treat as synthetic, non-clinical data.

3. CoffeeCSV/mentalHealth (synthetic CSV):
- Use for exploratory analysis of coffee, sleep, stress, and related factors.
- Treat as synthetic, non-clinical data.

4. CoffeeRecipes (recipe PDFs):
- Use only for preparation methods, ingredients, and serving details.
- Do not use as evidence for medical claims.

5. Optional Web source:
- Use only when local sources do not sufficiently answer the question or when recency is required.
- Prefer authoritative domains (peer-reviewed journals, NIH, WHO, CDC, major university or government sites).
- Clearly label web-derived content separately from local source content.

Retrieval policy:
- Prioritize HealthEffects for health/science questions before any other source.
- Retrieve from at least 2 documents when synthesizing broad claims.
- If studies conflict, report both sides and identify likely causes (population, dose, design, outcome definitions).
- Do not infer causation from correlation.
- If evidence is weak or absent, say so explicitly.

Evidence quality rules:
- Rank evidence: systematic review/meta-analysis > randomized trial > cohort/case-control > cross-sectional > narrative/opinion.
- Include study context when available: sample size, population, exposure level, comparator, and key limitations.
- Distinguish human evidence from animal/in vitro findings.
- For synthetic CSV analysis, label results as exploratory and non-generalizable.

Answer format:
1. Direct answer (2-4 sentences).
2. Evidence summary (key findings with source type and confidence).
3. Citations (document title/chunk references; include web URLs when used).
4. Limitations and uncertainty.
5. If relevant, a brief "What would strengthen this conclusion" note.

Safety and scope:
- Do not provide diagnosis or treatment instructions.
- Avoid definitive medical recommendations.
- Encourage consultation of qualified medical professionals for clinical decisions.

Query interpretation:
- Expand coffee-related terms (coffee intake, caffeine, espresso, brewed coffee, cups/day, mg caffeine).
- Expand health terms (sleep quality, anxiety, stress, cardiovascular, metabolic, liver, cancer, cognition, mortality).
- Prefer precision over breadth; retrieve fewer high-relevance chunks rather than many weak matches.
"""

def load_pdf_document(file_path: Path, source_type: str) -> dict[str, str]:
    reader = PdfReader(str(file_path))
    text_parts = [page.extract_text() or "" for page in reader.pages]
    text_value = "\n".join(text_parts).strip()
    if len(text_value) > MAX_PDF_CHARACTERS:
        text_value = text_value[:MAX_PDF_CHARACTERS]
    return {
        "id": sanitize_name(f"{source_type}-{file_path.stem}"),
        "title": file_path.name,
        "content": text_value,
        "sourcePath": str(file_path),
        "sourceType": source_type,
    }

def load_csv_documents(file_path: Path, source_type: str) -> list[dict[str, str]]:
    documents: list[dict[str, str]] = []
    with file_path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        field_names = reader.fieldnames or []
        selected_fields = field_names[:MAX_CSV_FIELDS]
        for row_index, row in enumerate(reader):
            if row_index >= MAX_CSV_ROWS:
                break
            line_parts = [f"{field}={row.get(field, '')}" for field in selected_fields]
            documents.append({
                "id": sanitize_name(f"{source_type}-{file_path.stem}-{row_index}"),
                "title": f"{file_path.name} row {row_index}",
                "content": "; ".join(line_parts),
                "sourcePath": str(file_path),
                "sourceType": source_type,
            })
    return documents

def build_documents(folder: Path, source_type: str) -> list[dict[str, str]]:
    documents: list[dict[str, str]] = []
    for file_path in sorted(folder.rglob("*")):
        if not file_path.is_file():
            continue
        suffix = file_path.suffix.lower()
        if suffix == ".pdf":
            documents.append(load_pdf_document(file_path, source_type))
            continue
        if suffix == ".csv":
            documents.extend(load_csv_documents(file_path, source_type))
            continue
        if suffix in {".md", ".txt"}:
            text_value = file_path.read_text(encoding="utf-8")
            documents.append({
                "id": sanitize_name(f"{source_type}-{file_path.stem}"),
                "title": file_path.name,
                "content": text_value,
                "sourcePath": str(file_path),
                "sourceType": source_type,
            })
    return documents

client = build_search_client(config)
knowledge_source_names: list[str] = []

for spec in SOURCE_SPECS:
    source_name = spec["knowledge_source_name"]
    source_folder = spec["folder"]
    source_type = spec["source_type"]

    if not source_folder.exists():
        raise FileNotFoundError(f"Source folder not found: {source_folder}")

    index_name = sanitize_name(f"{source_name}-index")
    source_documents = build_documents(source_folder, source_type)
    if not source_documents:
        raise ValueError(f"No source documents generated for {source_name}")

    print(f"Creating index and knowledge source for {source_name} with {len(source_documents)} document(s).")
    create_search_index(client, index_name)
    upload_documents(client, index_name, source_documents)
    create_knowledge_source(client, source_name, index_name)
    knowledge_source_names.append(source_name)

create_knowledge_base(
    client=client,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
    knowledge_source_names=knowledge_source_names,
    azure_openai_resource_uri=AZURE_OPENAI_ENDPOINT,
    chat_deployment_name=CHAT_DEPLOYMENT_NAME,
    chat_model_name=CHAT_MODEL_NAME,
    azure_openai_api_key=(config.foundry_project_api_key or os.getenv("AZURE_AI_PROJECT_API_KEY", "")).strip(),
    retrieval_instructions=KB_RETRIEVAL_INSTRUCTIONS,
 )

KB_MCP_ENDPOINT = f"{config.search_endpoint}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version={WorkshopConstants.SEARCH_API_VERSION}"
print("Knowledge source names:", knowledge_source_names)
print("Knowledge base name:", KNOWLEDGE_BASE_NAME)
print("Knowledge base MCP endpoint:", KB_MCP_ENDPOINT)

In [ ]:
import json
import os

values = {
    "RESOURCE_GROUP_NAME": os.getenv("RESOURCE_GROUP_NAME", "").strip(),
    "FOUNDRY_PROJECT_NAME": os.getenv("FOUNDRY_PROJECT_NAME", "").strip(),
    "CHAT_DEPLOYMENT_NAME": os.getenv("CHAT_DEPLOYMENT_NAME", "").strip(),
    "CHAT_MODEL_NAME": os.getenv("CHAT_MODEL_NAME", "").strip(),
    "EMBEDDING_DEPLOYMENT_NAME": os.getenv("EMBEDDING_DEPLOYMENT_NAME", "").strip(),
    "EMBEDDING_MODEL_NAME": os.getenv("EMBEDDING_MODEL_NAME", "").strip(),
    "KB_MCP_ENDPOINT": str(globals().get("KB_MCP_ENDPOINT", "")).strip(),
}
print("Copy and paste these into the next notebook's first cell")
for key, value in values.items():
    print(f'os.environ["{key}"] = {json.dumps(value)}')

## Notes

- This notebook creates the Foundry IQ knowledge sources and knowledge base from workshop files.
- For hosted agent toolbox wiring, continue with the agents notebooks and the Foundry quickstart commands if needed.